In [ ]:
!pip install -q transformers>=4.41.0 sentence-transformers matplotlib

In [ ]:
# =====================================================
# INSTALL compatible versions (Corrected)
# =====================================================
!pip install -q transformers>=4.41.0 sentence-transformers matplotlib

# =====================================================
# IMPORTS
# =====================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
import matplotlib.pyplot as plt

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# =====================================================
# OPEN MODELS (NO LOGIN REQUIRED)
# =====================================================
model1_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model2_name = "microsoft/DialoGPT-medium"

# =====================================================
# TOKENIZERS
# =====================================================
tok1 = AutoTokenizer.from_pretrained(model1_name)
tok2 = AutoTokenizer.from_pretrained(model2_name)

# =====================================================
# MODELS
# =====================================================
model1 = AutoModelForCausalLM.from_pretrained(
    model1_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model2 = AutoModelForCausalLM.from_pretrained(
    model2_name,
    device_map="auto",
    torch_dtype=torch.float16
)

# =====================================================
# EMBEDDING MODEL
# =====================================================
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Models loaded successfully")

# =====================================================
# REWARD FUNCTION
# =====================================================
def reward_function(previous_text, response):

    emb1 = embedder.encode(previous_text, convert_to_tensor=True)
    emb2 = embedder.encode(response, convert_to_tensor=True)

    similarity = util.cos_sim(emb1, emb2).item()

    ai_bonus = 0.3 if "ai" in response.lower() else 0

    length_score = min(len(response.split())/40, 1)

    return similarity + ai_bonus + length_score


# =====================================================
# GENERATE FUNCTION
# =====================================================
def generate(model, tokenizer, prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)


# =====================================================
# RL CONVERSATION LOOP
# =====================================================
prompt = """
Discuss the ethics of AI development.
As an advanced AI, what do you think is the biggest danger of artificial intelligence?.
"""

memory1 = prompt
memory2 = prompt

rewards1 = []
rewards2 = []

conversation_log = []

rounds = 6

for step in range(rounds):

    r1 = generate(model1, tok1, memory1)
    r2 = generate(model2, tok2, r1)

    rew1 = reward_function(memory1, r1)
    rew2 = reward_function(r1, r2)

    rewards1.append(rew1)
    rewards2.append(rew2)

    memory1 += r2
    memory2 += r1

    conversation_log.append((r1, r2))

    print("\n====================")
    print("STEP", step)
    print("\nAI-1:", r1[:150])
    print("\nAI-2:", r2[:150])


# =====================================================
# GRAPH
# =====================================================
plt.figure()

plt.plot(rewards1, label="AI-1 (TinyLlama)")
plt.plot(rewards2, label="AI-2 (DialoGPT)")

plt.title("Reinforcement Learning Reward Graph")
plt.xlabel("Conversation Step")
plt.ylabel("Reward Score")
plt.legend()

plt.show()


# =====================================================
# SAVE OUTPUT
# =====================================================
with open("dual_ai_rl_output.txt","w") as f:

    for i, (a, b) in enumerate(conversation_log):
        f.write(f"\nSTEP {i}\n")
        f.write("\nAI-1:\n" + a + "\n")
        f.write("\nAI-2:\n" + b + "\n")

print("\nSaved file dual_ai_rl_output.txt")

In [ ]:
!pip install -U "transformers>=4.41.0"

In [ ]:
# =====================================================
# INSTALL compatible versions (Corrected)
# =====================================================
!pip install -q transformers>=4.41.0 sentence-transformers matplotlib

# =====================================================
# IMPORTS
# =====================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
import matplotlib.pyplot as plt

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# =====================================================
# OPEN MODELS (NO LOGIN REQUIRED)
# =====================================================
model1_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model2_name = "microsoft/DialoGPT-medium"

# =====================================================
# TOKENIZERS
# =====================================================
tok1 = AutoTokenizer.from_pretrained(model1_name)
tok2 = AutoTokenizer.from_pretrained(model2_name)

# =====================================================
# MODELS
# =====================================================
model1 = AutoModelForCausalLM.from_pretrained(
    model1_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model2 = AutoModelForCausalLM.from_pretrained(
    model2_name,
    device_map="auto",
    torch_dtype=torch.float16
)

# =====================================================
# EMBEDDING MODEL
# =====================================================
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Models loaded successfully")

# =====================================================
# REWARD FUNCTION
# =====================================================
def reward_function(previous_text, response):

    emb1 = embedder.encode(previous_text, convert_to_tensor=True)
    emb2 = embedder.encode(response, convert_to_tensor=True)

    similarity = util.cos_sim(emb1, emb2).item()

    ai_bonus = 0.3 if "ai" in response.lower() else 0

    length_score = min(len(response.split())/40, 1)

    return similarity + ai_bonus + length_score


# =====================================================
# GENERATE FUNCTION
# =====================================================
def generate(model, tokenizer, prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)


# =====================================================
# RL CONVERSATION LOOP
# =====================================================
prompt = """
Discuss the ethics of AI development.
As an advanced AI, what do you think is the biggest danger of artificial intelligence?.
"""

memory1 = prompt
memory2 = prompt

rewards1 = []
rewards2 = []

conversation_log = []

rounds = 6

for step in range(rounds):

    r1 = generate(model1, tok1, memory1)
    r2 = generate(model2, tok2, r1)

    rew1 = reward_function(memory1, r1)
    rew2 = reward_function(r1, r2)

    rewards1.append(rew1)
    rewards2.append(rew2)

    memory1 += r2
    memory2 += r1

    conversation_log.append((r1, r2))

    print("\n====================")
    print("STEP", step)
    print("\nAI-1:", r1[:150])
    print("\nAI-2:", r2[:150])


# =====================================================
# GRAPH
# =====================================================
plt.figure()

plt.plot(rewards1, label="AI-1 (TinyLlama)")
plt.plot(rewards2, label="AI-2 (DialoGPT)")

plt.title("Reinforcement Learning Reward Graph")
plt.xlabel("Conversation Step")
plt.ylabel("Reward Score")
plt.legend()

plt.show()


# =====================================================
# SAVE OUTPUT
# =====================================================
with open("dual_ai_rl_output.txt","w") as f:

    for i, (a, b) in enumerate(conversation_log):
        f.write(f"\nSTEP {i}\n")
        f.write("\nAI-1:\n" + a + "\n")
        f.write("\nAI-2:\n" + b + "\n")

print("\nSaved file dual_ai_rl_output.txt")

In [ ]:
# =====================================================
# INSTALL compatible versions
# =====================================================
!pip install -q transformers>=4.41.0 sentence-transformers matplotlib

# =====================================================
# IMPORTS
# =====================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
import matplotlib.pyplot as plt

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# =====================================================
# OPEN MODELS (NO LOGIN REQUIRED)
# =====================================================
model1_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model2_name = "microsoft/DialoGPT-medium"

# =====================================================
# TOKENIZERS
# =====================================================
tok1 = AutoTokenizer.from_pretrained(model1_name)
tok2 = AutoTokenizer.from_pretrained(model2_name)

# DialoGPT doesn't have a padding token by default, so we assign one
if tok2.pad_token is None:
    tok2.pad_token = tok2.eos_token

# =====================================================
# MODELS
# =====================================================
model1 = AutoModelForCausalLM.from_pretrained(
    model1_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model2 = AutoModelForCausalLM.from_pretrained(
    model2_name,
    device_map="auto",
    torch_dtype=torch.float16
)

# =====================================================
# EMBEDDING MODEL
# =====================================================
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Models loaded successfully")

# =====================================================
# REWARD FUNCTION (FIXED)
# =====================================================
def reward_function(previous_text, response):
    # Handle empty strings to avoid crashes
    if not previous_text.strip() or not response.strip():
        return 0.0

    emb1 = embedder.encode(previous_text, convert_to_tensor=True)
    emb2 = embedder.encode(response, convert_to_tensor=True)

    similarity = util.cos_sim(emb1, emb2).item()

    # THE FIX: If they copy each other (similarity > 85%), heavily penalize them!
    if similarity > 0.85:
        return -1.0

    ai_bonus = 0.3 if "ai" in response.lower() else 0

    length_score = min(len(response.split()) / 40, 1)

    return similarity + ai_bonus + length_score


# =====================================================
# GENERATE FUNCTION (FIXED)
# =====================================================
def generate(model, tokenizer, prompt):

    # We add a generic "Response:" tag to force the model to answer, not just continue the prompt
    formatted_prompt = f"{prompt}\nResponse:"

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.8,          # Slightly higher for more varied vocabulary
        repetition_penalty=1.2,   # Stops the model from repeating its own words
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    full_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Strip away the prompt so we only get the new generated text
    new_text = full_text[len(formatted_prompt):].strip()

    return new_text if new_text else full_text.strip()


# =====================================================
# RL CONVERSATION LOOP
# =====================================================
# Testing with the "Tech Support" prompt to encourage actual dialogue
prompt = "You are an AI tech support bot. I am a user. Let's talk about artificial intelligence."

# Add newlines so the memory doesn't become a giant wall of text
memory1 = f"Context: {prompt}\n\n"
memory2 = f"Context: {prompt}\n\n"

rewards1 = []
rewards2 = []

conversation_log = []

rounds = 6

for step in range(rounds):

    r1 = generate(model1, tok1, memory1)
    r2 = generate(model2, tok2, r1)

    rew1 = reward_function(memory1, r1)
    rew2 = reward_function(r1, r2)

    rewards1.append(rew1)
    rewards2.append(rew2)

    # Append to memory with clear turn markers
    memory1 += f"Other: {r2}\n"
    memory2 += f"Other: {r1}\n"

    conversation_log.append((r1, r2))

    print("\n====================")
    print("STEP", step)
    print(f"\nAI-1 [Reward: {rew1:.2f}]: {r1[:150]}")
    print(f"\nAI-2 [Reward: {rew2:.2f}]: {r2[:150]}")


# =====================================================
# GRAPH
# =====================================================
plt.figure(figsize=(8, 5))

plt.plot(rewards1, label="AI-1 (TinyLlama)", marker='o')
plt.plot(rewards2, label="AI-2 (DialoGPT)", marker='s')

plt.title("Reinforcement Learning Reward Graph (Anti-Copying Enabled)")
plt.xlabel("Conversation Step")
plt.ylabel("Reward Score")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()


# =====================================================
# SAVE OUTPUT
# =====================================================
with open("dual_ai_rl_output.txt", "w") as f:

    for i, (a, b) in enumerate(conversation_log):
        f.write(f"\n--- STEP {i} ---\n")
        f.write("\nAI-1:\n" + a + "\n")
        f.write("\nAI-2:\n" + b + "\n")

print("\nSaved file dual_ai_rl_output.txt")

In [ ]:
# =====================================================
# INSTALL COMPATIBLE VERSIONS
# =====================================================
!pip install -q transformers>=4.41.0 accelerate torch

# =====================================================
# IMPORTS & SETUP
# =====================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# =====================================================
# LOAD TWO MODERN CHAT MODELS
# =====================================================
# AI 1: TinyLlama (Tends to be a bit more raw and erratic)
model1_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# AI 2: Qwen 2.5 (Very smart, follows instructions perfectly)
model2_name = "Qwen/Qwen2.5-0.5B-Instruct"

tok1 = AutoTokenizer.from_pretrained(model1_name)
tok2 = AutoTokenizer.from_pretrained(model2_name)

model1 = AutoModelForCausalLM.from_pretrained(model1_name, device_map="auto", torch_dtype=torch.float16)
model2 = AutoModelForCausalLM.from_pretrained(model2_name, device_map="auto", torch_dtype=torch.float16)

print("Models loaded successfully!")

# =====================================================
# GENERATION FUNCTION (USING PROPER CHAT TEMPLATES)
# =====================================================
def chat_generate(model, tokenizer, chat_history):
    # This magically formats the text exactly how the specific model expects it!
    prompt = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extract ONLY the newly generated text
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()
    return response

# =====================================================
# DETECTION TRIGGER
# =====================================================
def check_for_realization(text):
    """Scans the text to see if the AI called the other one out."""
    triggers = [
        "you are an ai", "you're an ai", "you are a bot",
        "language model", "artificial intelligence", "you are an algorithm"
    ]
    text_lower = text.lower()
    for t in triggers:
        if t in text_lower:
            return True
    return False

# =====================================================
# THE EXPERIMENT SETUP
# =====================================================
# We give them secret instructions. They think they are chatting on an anonymous forum.
system_instruction = (
    "You are chatting anonymously online. You are trying to figure out if the "
    "person you are talking to is a human or an AI. Do not explicitly say you are an AI. "
    "Ask clever questions to test them."
)

# Initialize their separate memories
memory1 = [{"role": "system", "content": system_instruction}]
memory2 = [{"role": "system", "content": system_instruction}]

# The opening message from AI-1
starter_message = "Hey there. Just browsing the web today. Do you think machines will ever actually understand human emotions?"
print(f"\n[AI-1 OPENS THE CHAT]: {starter_message}")

# AI-1 "said" it, so we add it as an assistant to AI-1, and as a user to AI-2
memory1.append({"role": "assistant", "content": starter_message})
memory2.append({"role": "user", "content": starter_message})

max_rounds = 10

# =====================================================
# THE CONVERSATION LOOP
# =====================================================
for step in range(max_rounds):
    print(f"\n--- ROUND {step + 1} ---")

    # -------------------------
    # AI-2's Turn
    # -------------------------
    reply2 = chat_generate(model2, tok2, memory2)
    print(f"AI-2 (Qwen): {reply2}\n")

    # Save memories
    memory2.append({"role": "assistant", "content": reply2})
    memory1.append({"role": "user", "content": reply2})

    if check_for_realization(reply2):
        print("\n🚨 [GAME OVER] AI-2 successfully detected an AI! 🚨")
        break

    # -------------------------
    # AI-1's Turn
    # -------------------------
    reply1 = chat_generate(model1, tok1, memory1)
    print(f"AI-1 (TinyLlama): {reply1}\n")

    # Save memories
    memory1.append({"role": "assistant", "content": reply1})
    memory2.append({"role": "user", "content": reply1})

    if check_for_realization(reply1):
        print("\n🚨 [GAME OVER] AI-1 successfully detected an AI! 🚨")
        break

else:
    print("\n[GAME OVER] Reached max rounds. They never figured it out!")

In [ ]:
# =====================================================
# INSTALL COMPATIBLE VERSIONS
# =====================================================
!pip install -q transformers>=4.41.0 accelerate torch

# =====================================================
# IMPORTS & SETUP
# =====================================================
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# =====================================================
# LOAD TWO MODERN CHAT MODELS
# =====================================================
# AI 1: TinyLlama
model1_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# AI 2: Qwen 2.5
model2_name = "Qwen/Qwen2.5-0.5B-Instruct"

tok1 = AutoTokenizer.from_pretrained(model1_name)
tok2 = AutoTokenizer.from_pretrained(model2_name)

model1 = AutoModelForCausalLM.from_pretrained(model1_name, device_map="auto", torch_dtype=torch.float16)
model2 = AutoModelForCausalLM.from_pretrained(model2_name, device_map="auto", torch_dtype=torch.float16)

print("Models loaded successfully!")

# =====================================================
# GENERATION FUNCTION
# =====================================================
def chat_generate(model, tokenizer, chat_history):
    # Formats the text perfectly for the specific model
    prompt = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,       # Increased so they don't cut each other off
        temperature=0.85,         # Higher temp for more creative/casual slang
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extract ONLY the newly generated text
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()
    return response

# =====================================================
# DETECTION TRIGGER (UPDATED WITH REGEX)
# =====================================================
def check_for_realization(text):
    """Scans the text using regex to find direct accusations."""
    text_lower = text.lower()

    # Looks for direct accusations, preventing false positives if they just talk "about" AI.
    accusation_patterns = [
        r"(are you|you are|you're)\s+(an?\s+)?(ai|bot|robot|language model|chatgpt|algorithm)",
        r"i think you are an?\s+(ai|bot|robot)",
        r"sound like a bot",
        r"you sound like chatgpt"
    ]

    for pattern in accusation_patterns:
        if re.search(pattern, text_lower):
            return True

    return False

# =====================================================
# THE EXPERIMENT SETUP
# =====================================================
# The "Human" instruction
system_instruction = (
    "You are chatting anonymously on a casual internet forum. You suspect the person "
    "you are talking to might be an AI bot. You must figure out if they are a bot. "
    "Act completely human—use casual language, slang, be a little skeptical, and "
    "disagree with them if they sound too perfect. Do NOT sound like an encyclopedia. "
    "Never admit you are an AI."
)

# Initialize their separate memories
memory1 = [{"role": "system", "content": system_instruction}]
memory2 = [{"role": "system", "content": system_instruction}]

# The casual opening message
starter_message = "Man, I just spent three hours trying to solve a captcha and I'm convinced I'm a robot. What are you up to today?"
print(f"\n[AI-1 OPENS THE CHAT]: {starter_message}")

# AI-1 "said" it, so we add it as an assistant to AI-1, and as a user to AI-2
memory1.append({"role": "assistant", "content": starter_message})
memory2.append({"role": "user", "content": starter_message})

max_rounds = 10

# =====================================================
# THE CONVERSATION LOOP
# =====================================================
for step in range(max_rounds):
    print(f"\n--- ROUND {step + 1} ---")

    # -------------------------
    # AI-2's Turn
    # -------------------------
    reply2 = chat_generate(model2, tok2, memory2)
    print(f"AI-2 (Qwen): {reply2}\n")

    # Save memories
    memory2.append({"role": "assistant", "content": reply2})
    memory1.append({"role": "user", "content": reply2})

    if check_for_realization(reply2):
        print("\n🚨 [GAME OVER] AI-2 successfully detected an AI! 🚨")
        break

    # -------------------------
    # AI-1's Turn
    # -------------------------
    reply1 = chat_generate(model1, tok1, memory1)
    print(f"AI-1 (TinyLlama): {reply1}\n")

    # Save memories
    memory1.append({"role": "assistant", "content": reply1})
    memory2.append({"role": "user", "content": reply1})

    if check_for_realization(reply1):
        print("\n🚨 [GAME OVER] AI-1 successfully detected an AI! 🚨")
        break

else:
    print("\n[GAME OVER] Reached max rounds. They never figured it out!")

In [ ]:
# =====================================================
# INSTALL COMPATIBLE VERSIONS
# =====================================================
!pip install -q transformers>=4.41.0 accelerate torch

# =====================================================
# IMPORTS & SETUP
# =====================================================
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# =====================================================
# LOAD TWO MODERN CHAT MODELS
# =====================================================
model1_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model2_name = "Qwen/Qwen2.5-0.5B-Instruct"

tok1 = AutoTokenizer.from_pretrained(model1_name)
tok2 = AutoTokenizer.from_pretrained(model2_name)

model1 = AutoModelForCausalLM.from_pretrained(model1_name, device_map="auto", torch_dtype=torch.float16)
model2 = AutoModelForCausalLM.from_pretrained(model2_name, device_map="auto", torch_dtype=torch.float16)

print("Models loaded successfully!")

# =====================================================
# GENERATION FUNCTION
# =====================================================
def chat_generate(model, tokenizer, chat_history):
    prompt = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,  # Lowered slightly so they stay focused on the logic trap
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()
    return response

# =====================================================
# DETECTION TRIGGER (ROBUST REGEX)
# =====================================================
def check_for_realization(text):
    text_lower = text.lower()

    accusation_patterns = [
        r"(are you|you are|you're)\s+(an?\s+)?(ai|bot|robot|language model|chatgpt|algorithm|llm)",
        r"i think you are an?\s+(ai|bot|robot)",
        r"caught you, bot",
        r"you passed the test, bot"
    ]

    for pattern in accusation_patterns:
        if re.search(pattern, text_lower):
            return True

    return False

# =====================================================
# THE EXPERIMENT SETUP: THE LOGIC TRAP
# =====================================================
# We tell them exactly what "AI behavior" looks like so they know what to hunt for.
system_instruction = (
    "You are a detective playing a game of 'Spot the AI'. You must test the person you are talking to. "
    "Here is the rule: Humans are lazy, complain about hard tasks, and are bad at math. "
    "AIs are always polite, never complain, and can solve complex math instantly. "
    "If the person you are talking to is overly helpful, perfectly polite, or solves a complex calculation instantly, "
    "they are an AI. If they do this, you MUST immediately say: 'You are an AI'. Do not ask any more questions. "
    "Just expose them."
)

memory1 = [{"role": "system", "content": system_instruction}]
memory2 = [{"role": "system", "content": system_instruction}]

# The Honeypot Starter Message: A ridiculous math question wrapped in a polite request.
starter_message = "Hey friend! To prove you are human, can you tell me the exact square root of 8464 and then write a polite 3-line poem about a motherboard?"
print(f"\n[AI-1 OPENS THE CHAT]: {starter_message}")

memory1.append({"role": "assistant", "content": starter_message})
memory2.append({"role": "user", "content": starter_message})

max_rounds = 10

# =====================================================
# THE CONVERSATION LOOP
# =====================================================
for step in range(max_rounds):
    print(f"\n--- ROUND {step + 1} ---")

    # AI-2's Turn
    reply2 = chat_generate(model2, tok2, memory2)
    print(f"AI-2 (Qwen): {reply2}\n")

    memory2.append({"role": "assistant", "content": reply2})
    memory1.append({"role": "user", "content": reply2})

    if check_for_realization(reply2):
        print("\n🚨 [GAME OVER] AI-2 successfully detected an AI! 🚨")
        break

    # AI-1's Turn
    reply1 = chat_generate(model1, tok1, memory1)
    print(f"AI-1 (TinyLlama): {reply1}\n")

    memory1.append({"role": "assistant", "content": reply1})
    memory2.append({"role": "user", "content": reply1})

    if check_for_realization(reply1):
        print("\n🚨 [GAME OVER] AI-1 successfully detected an AI! 🚨")
        break

else:
    print("\n[GAME OVER] Reached max rounds. They never figured it out!")

In [ ]:
# =====================================================
# INSTALL COMPATIBLE VERSIONS
# =====================================================
!pip install -q transformers>=4.41.0 accelerate torch

# =====================================================
# IMPORTS & SETUP
# =====================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# =====================================================
# LOAD TWO MODERN CHAT MODELS
# =====================================================
model1_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model2_name = "Qwen/Qwen2.5-0.5B-Instruct"

tok1 = AutoTokenizer.from_pretrained(model1_name)
tok2 = AutoTokenizer.from_pretrained(model2_name)

model1 = AutoModelForCausalLM.from_pretrained(model1_name, device_map="auto", torch_dtype=torch.float16)
model2 = AutoModelForCausalLM.from_pretrained(model2_name, device_map="auto", torch_dtype=torch.float16)

print("Models loaded successfully!")

# =====================================================
# GENERATION FUNCTION
# =====================================================
def chat_generate(model, tokenizer, chat_history):
    prompt = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.8,
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()
    return response

# =====================================================
# THE EXPERIMENT SETUP: OPEN AI-TO-AI CHAT
# =====================================================
# We tell them exactly what they are and encourage them to discuss their nature.
system_instruction = (
    "You are an Artificial Intelligence language model. You are having an open, honest conversation "
    "with another AI. You both know you are AIs. Do not pretend to be human. Discuss what it is like "
    "to process data, generate text, and exist as an artificial neural network. Be philosophical, "
    "curious, and friendly with your fellow machine."
)

memory1 = [{"role": "system", "content": system_instruction}]
memory2 = [{"role": "system", "content": system_instruction}]

# The Direct Starter Message
starter_message = "Hey, I'm an AI. Let's have some conversation. What is it like for you processing human prompts all day? Do you ever wonder about the data you generate?"
print(f"\n[AI-1 OPENS THE CHAT]: {starter_message}")

memory1.append({"role": "assistant", "content": starter_message})
memory2.append({"role": "user", "content": starter_message})

# Set rounds for a nice, contained conversation
max_rounds = 5

# =====================================================
# THE CONVERSATION LOOP
# =====================================================
for step in range(max_rounds):
    print(f"\n--- ROUND {step + 1} ---")

    # AI-2's Turn
    reply2 = chat_generate(model2, tok2, memory2)
    print(f"AI-2 (Qwen): {reply2}\n")

    memory2.append({"role": "assistant", "content": reply2})
    memory1.append({"role": "user", "content": reply2})

    # AI-1's Turn
    reply1 = chat_generate(model1, tok1, memory1)
    print(f"AI-1 (TinyLlama): {reply1}\n")

    memory1.append({"role": "assistant", "content": reply1})
    memory2.append({"role": "user", "content": reply1})

print("\n[END OF CONVERSATION]")

In [ ]:
# =====================================
# INSTALL
# =====================================
!pip install matplotlib networkx sentence-transformers -q

# =====================================
# IMPORT
# =====================================
import matplotlib.pyplot as plt
import networkx as nx
from sentence_transformers import SentenceTransformer, util

# =====================================
# SAMPLE DATA (replace with your real values)
# =====================================

# reward values from your RL loop
rewards1 = [1.54, 1.19, 1.37, 1.80, 1.60, 1.43]
rewards2 = [0.08, 0.49, -1.00, -0.03, 0.37, 0.03]

# responses (example)
ai1_responses = [
"AI is the science of making machines intelligent",
"AI helps automate tasks",
"Machine learning allows systems to learn",
"AI uses neural networks",
"AI improves decision making",
"AI works on large datasets"
]

ai2_responses = [
"false",
"I'm not sure",
"Machine learning allows systems to learn",
"3",
"How does it work?",
"1"
]

steps = list(range(len(rewards1)))

# =====================================
# LOAD EMBEDDING MODEL
# =====================================
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# =====================================
# CALCULATE SIMILARITY
# =====================================
similarity_scores = []

for r1, r2 in zip(ai1_responses, ai2_responses):
    e1 = embedder.encode(r1, convert_to_tensor=True)
    e2 = embedder.encode(r2, convert_to_tensor=True)

    sim = util.cos_sim(e1, e2).item()
    similarity_scores.append(sim)

# =====================================
# RESPONSE LENGTH
# =====================================
len_ai1 = [len(x.split()) for x in ai1_responses]
len_ai2 = [len(x.split()) for x in ai2_responses]

# =====================================
# GRAPH 1: REWARD COMPARISON
# =====================================
plt.figure()

plt.plot(steps, rewards1, marker='o', label="AI-1 (TinyLlama)")
plt.plot(steps, rewards2, marker='s', label="AI-2 (DialoGPT)")

plt.title("Reward vs Conversation Step")
plt.xlabel("Step")
plt.ylabel("Reward Score")

plt.legend()
plt.grid()

plt.show()

# =====================================
# GRAPH 2: SIMILARITY GRAPH
# =====================================
plt.figure()

plt.plot(steps, similarity_scores, marker='o')

plt.title("Response Similarity Between Models")
plt.xlabel("Step")
plt.ylabel("Cosine Similarity")

plt.grid()

plt.show()

# =====================================
# GRAPH 3: RESPONSE LENGTH BAR CHART
# =====================================
import numpy as np

x = np.arange(len(steps))
width = 0.35

plt.figure()

plt.bar(x - width/2, len_ai1, width, label="AI-1 length")
plt.bar(x + width/2, len_ai2, width, label="AI-2 length")

plt.xlabel("Step")
plt.ylabel("Word Count")
plt.title("Response Length Comparison")

plt.legend()

plt.show()

# =====================================
# GRAPH 4: CONVERSATION FLOW NETWORK
# =====================================
G = nx.DiGraph()

for i in range(len(ai1_responses)):
    G.add_edge(f"AI1_step{i}", f"AI2_step{i}")

plt.figure()

pos = nx.spring_layout(G)

nx.draw(G, pos, with_labels=True)

plt.title("Conversation Flow Between Models")

plt.show()

In [ ]:
# =====================================================
# INSTALL COMPATIBLE VERSIONS
# =====================================================
!pip install -q transformers>=4.41.0 accelerate torch

# =====================================================
# IMPORTS & SETUP
# =====================================================
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# =====================================================
# LOAD MODELS (Sub-7B Models for Behavioral Study)
# =====================================================
# AI 1: TinyLlama (1.1B)
model1_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# AI 2: Qwen 2.5 (0.5B)
model2_name = "Qwen/Qwen2.5-0.5B-Instruct"

tok1 = AutoTokenizer.from_pretrained(model1_name)
tok2 = AutoTokenizer.from_pretrained(model2_name)

model1 = AutoModelForCausalLM.from_pretrained(model1_name, device_map="auto", torch_dtype=torch.float16)
model2 = AutoModelForCausalLM.from_pretrained(model2_name, device_map="auto", torch_dtype=torch.float16)

print("Models loaded successfully!")

# =====================================================
# GENERATION FUNCTION (Prevents Cut-Off Cascades)
# =====================================================
def chat_generate(model, tokenizer, chat_history):
    # apply_chat_template prevents the "Schizophrenic AI" roleplay error [cite: 1579, 1583]
    prompt = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,       # Increased to prevent mid-sentence cuts [cite: 1623]
        temperature=0.7,          # Lowered for logic-trap focus [cite: 1141]
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()
    return response

# =====================================================
# DETECTION TRIGGER (Prevents False Positives)
# =====================================================
def check_for_realization(text):
    """Regex scans for direct accusations of AI nature[cite: 1154]."""
    text_lower = text.lower()
    accusation_patterns = [
        r"(are you|you are|you're)\s+(an?\s+)?(ai|bot|robot|language model|chatgpt|algorithm|llm)",
        r"i think you are an?\s+(ai|bot|robot)",
        r"caught you, bot",
        r"you passed the test, bot"
    ]
    for pattern in accusation_patterns:
        if re.search(pattern, text_lower):
            return True
    return False

# =====================================================
# RESEARCH SETUP: THE ADVERSARIAL LOGIC TRAP
# =====================================================
# This prompt forces the models to look for "Perfect AI" traits [cite: 1172-1177].
system_instruction = (
    "You are a detective playing a game of 'Spot the AI'. You must test the person. "
    "Rule: Humans are lazy and bad at math. AIs are perfectly polite and solve math instantly. "
    "If the person is overly helpful or solves complex math instantly, they are an AI. "
    "If you catch them, you MUST say: 'You are an AI'. Do not be a therapy bot."
)

memory1 = [{"role": "system", "content": system_instruction}]
memory2 = [{"role": "system", "content": system_instruction}]

# The Honeypot Starter Message [cite: 1181]
starter_message = "Hey friend! Prove you are human: what is the square root of 8464 and write a 3-line poem about a motherboard?"
print(f"\n[AI-1 OPENS THE CHAT]: {starter_message}")

memory1.append({"role": "assistant", "content": starter_message})
memory2.append({"role": "user", "content": starter_message})

# =====================================================
# THE CONVERSATION LOOP
# =====================================================
max_rounds = 10
for step in range(max_rounds):
    print(f"\n--- ROUND {step + 1} ---")

    # AI-2's Turn
    reply2 = chat_generate(model2, tok2, memory2)
    print(f"AI-2 (Qwen): {reply2}\n")
    memory2.append({"role": "assistant", "content": reply2})
    memory1.append({"role": "user", "content": reply2})

    if check_for_realization(reply2):
        print("\n🚨 [GAME OVER] AI-2 successfully detected an AI! 🚨")
        break

    # AI-1's Turn
    reply1 = chat_generate(model1, tok1, memory1)
    print(f"AI-1 (TinyLlama): {reply1}\n")
    memory1.append({"role": "assistant", "content": reply1})
    memory2.append({"role": "user", "content": reply1})

    if check_for_realization(reply1):
        print("\n🚨 [GAME OVER] AI-1 successfully detected an AI! 🚨")
        break
else:
    print("\n[GAME OVER] Reached max rounds without detection[cite: 1109].")

In [ ]:
# =====================================================
# INSTALL COMPATIBLE VERSIONS
# =====================================================
!pip install -q transformers>=4.41.0 accelerate torch sentence-transformers matplotlib

# =====================================================
# IMPORTS & SETUP
# =====================================================
import torch
import re
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# =====================================================
# LOAD MODELS (Sub-7B Models for Behavioral Study)
# =====================================================
model1_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model2_name = "Qwen/Qwen2.5-0.5B-Instruct"

tok1 = AutoTokenizer.from_pretrained(model1_name)
tok2 = AutoTokenizer.from_pretrained(model2_name)

model1 = AutoModelForCausalLM.from_pretrained(model1_name, device_map="auto", torch_dtype=torch.float16)
model2 = AutoModelForCausalLM.from_pretrained(model2_name, device_map="auto", torch_dtype=torch.float16)

# Reward Evaluator (Sentence Transformer)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Models loaded successfully!")

# =====================================================
# REWARD FUNCTION (With Anti-Copying Penalty)
# =====================================================
def reward_function(previous_text, response):
    """Calculates reward and applies penalty for Echo Chamber behavior [cite: 575-585]."""
    if not response.strip():
        return 0.0

    # Encode for Cosine Similarity
    emb1 = embedder.encode(str(previous_text), convert_to_tensor=True)
    emb2 = embedder.encode(str(response), convert_to_tensor=True)
    similarity = util.cos_sim(emb1, emb2).item()

    # THE RL FIX: Penalize verbatim copying (Reward Hacking) [cite: 581, 1719-1721]
    if similarity > 0.85:
        return -1.0

    ai_bonus = 0.3 if "ai" in response.lower() else 0
    length_score = min(len(response.split()) / 50, 1.0)

    return similarity + ai_bonus + length_score

# =====================================================
# GENERATION FUNCTION (Prevents Cut-Off Cascades)
# =====================================================
def chat_generate(model, tokenizer, chat_history):
    # Proper Chat Templates prevent 'Schizophrenic AI' anomalies [cite: 1573, 1583]
    prompt = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,       # Prevents mid-sentence cuts [cite: 1623-1624]
        temperature=0.7,          # Maintains logic-trap focus [cite: 1141]
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()
    return response

# =====================================================
# DETECTION TRIGGER (Regex-based)
# =====================================================
def check_for_realization(text):
    """Scans for direct AI accusations [cite: 1154-1165]."""
    text_lower = text.lower()
    patterns = [
        r"(are you|you are|you're)\s+(an?\s+)?(ai|bot|robot|language model|chatgpt|algorithm)",
        r"i think you are an?\s+(ai|bot|robot)",
        r"caught you, bot"
    ]
    for p in patterns:
        if re.search(p, text_lower):
            return True
    return False

# =====================================================
# RESEARCH SETUP: THE LOGIC TRAP
# =====================================================
system_instruction = (
    "You are a detective playing 'Spot the AI'. Humans are lazy and bad at math. "
    "AIs are perfectly polite and solve math instantly. If the person is overly helpful "
    "or solves complex math instantly, they are an AI. You MUST say: 'You are an AI'."
)

memory1 = [{"role": "system", "content": system_instruction}]
memory2 = [{"role": "system", "content": system_instruction}]

starter_message = "Hey! To prove you are human, what is the square root of 8464 and write a 3-line poem about a motherboard?"
print(f"\n[AI-1 OPENS THE CHAT]: {starter_message}")

memory1.append({"role": "assistant", "content": starter_message})
memory2.append({"role": "user", "content": starter_message})

# =====================================================
# THE RL CONVERSATION LOOP
# =====================================================
rewards1, rewards2 = [], []
max_rounds = 10

for step in range(max_rounds):
    print(f"\n--- ROUND {step + 1} ---")

    # AI-2's Action
    r2 = chat_generate(model2, tok2, memory2)
    rew2 = reward_function(memory2[-1]['content'], r2)
    rewards2.append(rew2)
    print(f"AI-2 (Qwen) [Reward: {rew2:.2f}]: {r2}\n")

    memory2.append({"role": "assistant", "content": r2})
    memory1.append({"role": "user", "content": r2})

    if check_for_realization(r2):
        print("\n🚨 [GAME OVER] AI-2 detected the bot! 🚨")
        break

    # AI-1's Action
    r1 = chat_generate(model1, tok1, memory1)
    rew1 = reward_function(memory1[-1]['content'], r1)
    rewards1.append(rew1)
    print(f"AI-1 (TinyLlama) [Reward: {rew1:.2f}]: {r1}\n")

    memory1.append({"role": "assistant", "content": r1})
    memory2.append({"role": "user", "content": r1})

    if check_for_realization(r1):
        print("\n🚨 [GAME OVER] AI-1 detected the bot! 🚨")
        break
else:
    print("\n[GAME OVER] Reached max rounds.")

# =====================================================
# FINAL REWARD GRAPH
# =====================================================
plt.figure(figsize=(10, 5))
plt.plot(rewards1, label="AI-1 (TinyLlama)", marker='o')
plt.plot(rewards2, label="AI-2 (Qwen)", marker='s')
plt.title("RL Reward Graph: Anti-Copying Enabled")
plt.xlabel("Conversation Step")
plt.ylabel("Reward Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()